In [1]:
# ===== COMMON SETUP =====

import pandas as pd
import numpy as np
import torch
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, multilabel_confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm

# Load data
df = pd.read_csv("train.csv")

# Preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["comment_text"] = df["comment_text"].astype(str).apply(clean_text)
df = df.dropna(subset=["comment_text"])

labels_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
df["labels"] = df[labels_cols].values.tolist()

# Reduce size for faster run
df = df.sample(8000, random_state=42)

# Split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Dataset
class ToxicDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

# Loaders
train_loader = DataLoader(ToxicDataset(train_df["comment_text"], train_df["labels"]), batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(ToxicDataset(val_df["comment_text"], val_df["labels"]), batch_size=16, num_workers=0)
test_loader = DataLoader(ToxicDataset(test_df["comment_text"], test_df["labels"]), batch_size=16, num_workers=0)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Metrics
def compute_metrics(preds, labels):
    preds = (preds > 0.5).astype(int)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="micro"),
        "recall": recall_score(labels, preds, average="micro"),
        "f1": f1_score(labels, preds, average="micro")
    }

def evaluate(model, loader):
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            preds.extend(torch.sigmoid(logits).cpu().numpy())
            true.extend(labels.cpu().numpy())

    return np.array(preds), np.array(true)

## 🔹 EXPERIMENT 1: FREEZE BERT

In [4]:
print("\n=== Experiment 1: Freeze BERT ===")

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=6,
    problem_type="multi_label_classification"
)

# Freeze BERT
for param in model.bert.parameters():
    param.requires_grad = False

model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

def train(model, loader):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

# Train
for epoch in range(2):
    loss = train(model, train_loader)
    preds, labels = evaluate(model, val_loader)
    print("Loss:", loss)
    print(compute_metrics(preds, labels))


=== Experiment 1: Freeze BERT ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 50/50 [00:10<00:00, 

Loss: 0.38030211329460145
{'accuracy': 0.91, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}


100%|██████████| 50/50 [00:10<00:00,  4.59it/s]

Loss: 0.1862229032628238
{'accuracy': 0.91, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}



C:\Users\anya2\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Exp 1 Result:

When BERT was frozen, the model failed to learn meaningful representations of toxic comments. Due to class imbalance, the model predicted all samples as non-toxic, resulting in high accuracy but zero precision, recall, and F1-score. This demonstrates that freezing BERT significantly reduces model performance.

## 🔹EXPERIMENT 2: LAST 2 LAYERS ONLY

In [3]:
print("\n=== Experiment 2: Fine-tune Last 2 Layers ===")

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=6,
    problem_type="multi_label_classification"
)

# Freeze all
for param in model.bert.parameters():
    param.requires_grad = False

# Unfreeze last 2 layers
for name, param in model.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True

model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

def train(model, loader):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

# Train
for epoch in range(2):
    loss = train(model, train_loader)
    preds, labels = evaluate(model, val_loader)
    print("Loss:", loss)
    print(compute_metrics(preds, labels))


=== Experiment 2: Fine-tune Last 2 Layers ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 50/50 [00:09<00:00, 

Loss: 0.18898032144643367
{'accuracy': 0.91625, 'precision': 0.8813559322033898, 'recall': 0.348993288590604, 'f1': 0.5}


100%|██████████| 50/50 [00:10<00:00,  5.00it/s]

Loss: 0.07204055261798203
{'accuracy': 0.915, 'precision': 0.736, 'recall': 0.6174496644295302, 'f1': 0.6715328467153284}


Exp 2 Results:

Fine-tuning only the last two layers of BERT significantly improved performance compared to the fully frozen model. The model was able to learn task-specific features, resulting in improved precision, recall, and F1-score. However, since most of the BERT layers were frozen, the model still had limited ability to capture deeper contextual representations.

## 🔹 EXPERIMENT 3: FULL FINE-TUNING

In [3]:
print("\n=== Experiment 3: Full Fine-tuning ===")

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=6,
    problem_type="multi_label_classification"
)

# Unfreeze all
for param in model.parameters():
    param.requires_grad = True

model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

def train(model, loader):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

# Train
for epoch in range(2):
    loss = train(model, train_loader)
    preds, labels = evaluate(model, val_loader)
    print("Loss:", loss)
    print(compute_metrics(preds, labels))


=== Experiment 3: Full Fine-tuning ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 50/50 [00:10<00:00, 

Loss: 0.11462676713010296
{'accuracy': 0.93, 'precision': 0.8902439024390244, 'recall': 0.4899328859060403, 'f1': 0.6320346320346321}


100%|██████████| 50/50 [00:10<00:00,  4.83it/s]

Loss: 0.04648530898732133
{'accuracy': 0.93, 'precision': 0.7857142857142857, 'recall': 0.6644295302013423, 'f1': 0.72}


Exp 3 Result:

As the number of trainable layers increases, the model performance improves significantly. Full fine-tuning of BERT provides the best balance between precision and recall, resulting in the highest F1-score.

## Final Results & Comparision

| Experiment        | Accuracy | Precision | Recall   | F1 Score |
| ----------------- | -------- | --------- | -------- | -------- |
| 🔹 Freeze BERT    | 0.91     | 0.00      | 0.00     | 0.00     |
| 🔹 Last 2 Layers  | 0.915    | 0.73      | 0.62     | 0.67     |
| 🔹 Full Fine-tune | **0.93** | **0.79**  | **0.66** | **0.72** |
